### 1. Qual é o tempo médio (e mediano) entre a distribuição do pedido de Recuperação Judicial (RJ) e a concessão do processamento (despacho inicial)?

A pergunta se refere ao tempo médio (e mediano) entre a distribuição do
pedido de Recuperação Judicial e a concessão do processamento - despacho inicial que instaura a RJ (art. 52 da Lei 11.101/2005) - e dispara a proteção do art. 6º, § 4º (início do stay period).

**Fonte:** `default.gold_base_analitica` (mart da camada Gold, criado pelo notebook 03_gold_modelagem).
A duração já vem materializada na coluna `dias_distribuicao_processamento`, calculada como `data_deferimento_rj − data_ajuizamento`. O presente notebook é consumidor: não reconstrói a semântica, apenas a consulta.

**Recorte:**
- `classe_codigo = 129` — classe CNJ "Recuperação Judicial";
- `escopo_lei_atual` — somente processos sob a Lei 11.101/2005 (exclui o legado da Lei 7.661/1945);
- `dias_distribuicao_processamento >= 0` — higiene temporal: exclui 7 processos (0,3% dos 2.422 com marco) cuja duração calculada ficou negativa por ruído de data na origem.

##### Método — como identificar a "concessão do processamento" da recuperação judicial nos dados

**O problema:** o ato existe na lei, mas não existe no vocabulário do dado.

A Lei 11.101/2005 prevê que o juiz, ao acolher o pedido, defere o
*processamento* da recuperação (art. 52) — despacho que instaura a RJ e
dispara a proteção do art. 6º, § 4º. Só que o padrão Datajud não tem um movimento próprio para esse ato: a TPU do CNJ não reserva código para "deferimento do processamento". O juiz o pratica dentro de movimentos
genéricos — às vezes como texto livre de um despacho, às vezes como uma decisão de deferimento codificada, às vezes invisível entre atos ordinatórios. O primeiro desafio metodológico do projeto foi, portanto, decidir como caracterizar um momento que o dado não nomeia.

**As três alternativas testadas (em ordem de confiabilidade):**

1. **Texto do despacho (regra estrita, R9 da camada gold)** — procurar, no nome do movimento e nos complementos, os padrões textuais do despacho processador ("processamento"). Essa seria a caracterização mais fiel: é o próprio ato. A cobertura, porém, é mínima (~1%) — na maioria dos tribunais o texto do despacho não é registrado como movimento.

2. **TPU 12444 — decisão de deferimento genérica** — um código que significa "deferiu-se algo". Não é o ato em si, mas é uma decisão (não um ato ordinatório) e, quando ocorre logo após a distribuição de uma RJ, é o candidato natural ao deferimento do processamento. Validado pela investigação da pergunta 2 (trajetória deferimento → concessão).

3. **Proxy "Mero expediente" (TPU 11010, janela 15–180 dias)** — o ato
   ordinatório mais comum da tramitação. A hipótese: o primeiro 11010
   entre 15 e 180 dias após a distribuição acompanha a tramitação do
   despacho processador. É o que dá cobertura — os demais capturam o
   ato "com nome"; este o infere pelo comportamento.

A cascata de regras formada por essas três alternativas - **R9 → 12444 → proxy** - resolve a convivência entre elas: a caracterização mais confiável vence; o proxy só preenche quem não tem marco estrito. Assim, o ato nunca é caracterizado por um método mais fraco quando um mais forte está disponível.

**Por que 15 e 180 dias são limites razoáveis — e não arbitrados no escuro.**
Os dois números foram escolhidos com fundamento e verificados por teste de sensibilidade:

- **Piso de 15 dias (defesa contra contaminação):** o 11010 (mero expediente) é um ato genérico que acompanha *toda* tramitação. Nos primeiros dias pós-distribuição, é mais provável que registre expediente de rotina (juntadas, intimações) do que um despacho processador. Um piso baixo  (ex.: 7 dias) aumentaria marginalmente a cobertura, mas injetaria falsos positivos com tempos "bons demais", puxando a mediana artificialmente para baixo.
- **Teto de 180 dias (defesa contra falso negativo):** passados 6 meses, um primeiro "mero expediente" tem pouca plausibilidade de acompanhar o despacho processador — ou o juízo demorou muito além do razoável, ou registra pouco. Incluir essa cauda arriscaria medir tramitação genérica,  não concessão.
- **A verificação empírica:** medindo o peso das bordas, apenas **2,1%** dos casos atingem o piso e **0,3%** o teto (**2,4%** somados); excluindo as bordas, a mediana move de 48 → 49 e a média de  127,3 → 125,6 — alteração marginal. Ou seja: a janela captura comportamento natural, não trunca a distribuição — a métrica não depende dos limites escolhidos.

O resultado do teste fecha o ciclo metodológico: os limites não foram impostos, foram arbitrados com fundamento e auditados com dado — o que blinda a escolha contra questionamento.


**Limitações:** 
1. O marco de concessão da RJ não existe na TPU — é caracterizado por cascata, com cobertura de 42,4% do corpus;

2. O proxy 11010 infere o ato pelo comportamento tramitatório — a janela 15–180 é validada por sensibilidade (2,4% de efeito de borda), mas ainda é inferência, não registro;

3. 7 casos (0,3%) têm datas incoerentes e são excluídos por higiene temporal;

4. A cobertura do marco é desigual entre tribunais — varas de maior volume registram menos (achado quantificado na pergunta 4).

#### O que a consulta a seguir mede
Esta consulta faz uma análise exploratória dos dados e mede o trecho
'ajuizamento → concessão' sobre os casos elegíveis, após a higiene temporal.

**Entrada:** `default.gold_base_analitica`, classe 129, escopo da lei
vigente, com higiene temporal (duração ≥ 0).

A consulta lê uma coluna por processo:

| Coluna lida | O que representa |
|---|---|
| `dias_distribuicao_processamento` | dias entre a distribuição e a concessão do processamento — materializada no Bloco 4 do notebook 03 pela cascata R9 → TPU 12444 → proxy 11010. Quando o campo é nulo significa sem marco identificável |

**Saída — ela produz uma única linha com as colunas a seguir:**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_rj` | conta os processos do recorte | o universo da pergunta (5.709) |
| `com_marco_identificavel` | conta os casos com concessão datada | quantos têm o marco identificável (2.422) |
| `excl_negativos` | conta os casos com duração < 0 | datas incoerentes na origem, excluídas por higiene temporal (7) |
| `qtd_processos` | conta os elegíveis (duração ≥ 0) | o "n" da fórmula estatística, após a higiene (2.415 = 2.422 − 7) |
| `minimo_dias` | mínimo da duração | o piso da distribuição após a higiene |
| `media_dias` | média da duração | o custo da cauda longa (127,3) |
| `mediana_dias` | percentil 50 | o tempo típico — a métrica central desta pergunta (48) |
| `p90_dias` | percentil 90 | o limiar além do qual estão os 10% mais demorados (149) |

**Em uma frase:** a consulta após fazer um levantamento dos dados, mede quanto tempo o sistema leva, na mediana e na cauda, para conceder o processamento da RJ — o marco que instaura a recuperação e dispara a proteção do art. 6º, § 4º.

In [0]:
%sql
-- Q1 · DURAÇÃO ATE A CONCESSAO: análise exploratória + estatisticas (nacional)
WITH base AS (
  SELECT numero_processo, dias_distribuicao_processamento
  FROM default.gold_base_analitica
  WHERE classe_codigo = 129 AND escopo_lei_atual
)
SELECT
  (SELECT count(*) FROM base)                                                     AS total_rj,
  (SELECT count(*) FROM base WHERE dias_distribuicao_processamento IS NOT NULL)   AS com_marco_identificavel,
  (SELECT count(*) FROM base WHERE dias_distribuicao_processamento < 0)           AS excl_negativos,
  count(*)                                                                        AS qtd_processos,
  round(min(dias_distribuicao_processamento), 1)                                  AS minimo_dias,
  round(avg(dias_distribuicao_processamento), 1)                                  AS media_dias,
  round(percentile_approx(dias_distribuicao_processamento, 0.5), 1)               AS mediana_dias,
  round(percentile_approx(dias_distribuicao_processamento, 0.9), 1)               AS p90_dias
FROM base
WHERE dias_distribuicao_processamento >= 0;

#### Análise da resposta

**Cobertura da métrica:** dos 5.709 processos no escopo da LRE (classe 129),
2.422 (42,4%) têm marco identificável de concessão — resultado da cascata
R9 → TPU 12444 → proxy 11010. Dos
2.422, 2.415 (99,7%) têm duração não-negativa e entram na estatística;
os 7 restantes (0,3%) carregam data inválida anterior ao ajuizamento e
são excluídos por higiene.

**Distribuição (2.415 processos elegíveis):**
| Estatística | Valor |
|---|---|
| Mediana | 48 dias |
| Média | 127,3 dias |
| P90 | 149 dias |


**Assimetria:** a média (127) é 2,6× a mediana (48) — distribuição
fortemente assimétrica à direita. Metade das concessões sai em ~7
semanas.  Por isso a leitura central desta
pergunta é a **mediana**, com a média documentando o custo da cauda.



#### O que a próxima consulta realiza (sensibilidade da janela 15–180)
A consulta faz medições para verificar a sensibilidade da janela 15-180 dias usada para considerar o primeiro movimento classificado como mero expediente (11010), como movimento que caracteriza a concessão da recuperação judicial.

**Entrada:** a mesma coluna da consulta anterior, agora **sem** o filtro
de higiene — o teste mede o peso das bordas sobre todas as concessões
com marco, isolando os valores cravados no piso (15) e no teto (180).

**Saída — o que ela produz (uma única linha com sete números):**

| Coluna produzida | Como é calculada | O que significa |
|---|---|---|
| `total_com_marco` | conta os processos com concessão datada | a base do teste (2.422) |
| `no_piso_15` / `no_teto_180` | conta os cravados em 15 e 180 | quantos casos a janela capturou na borda |
| `pct_piso` / `pct_teto` | contagens ÷ total × 100 | o peso das bordas (2,1% e 0,3%) |
| `media_sem_bordas` / `mediana_sem_bordas` | estatísticas excluindo 15 e 180 | o que restaria da métrica sem a janela |

**Em uma frase:** o teste remove as bordas da janela arbitrada e
recomputa a métrica — se mediana e média quase não se movem, a escolha
de 15–180 dias captura comportamento natural em vez de truncar a
distribuição.

In [0]:
%sql
-- Q1 · Peso das bordas da janela arbitrada + sensibilidade
SELECT
  count(*)                                                                          AS total_com_marco,
  count(CASE WHEN dias_distribuicao_processamento = 15  THEN 1 END)                 AS no_piso_15,
  count(CASE WHEN dias_distribuicao_processamento = 180 THEN 1 END)                 AS no_teto_180,
  round(count(CASE WHEN dias_distribuicao_processamento = 15  THEN 1 END) / count(*) * 100, 1) AS pct_piso,
  round(count(CASE WHEN dias_distribuicao_processamento = 180 THEN 1 END) / count(*) * 100, 1) AS pct_teto,
  round(avg(CASE WHEN dias_distribuicao_processamento NOT IN (15, 180)
             THEN dias_distribuicao_processamento END), 1)                          AS media_sem_bordas,
  round(percentile_approx(CASE WHEN dias_distribuicao_processamento NOT IN (15, 180)
             THEN dias_distribuicao_processamento END, 0.5), 1)                     AS mediana_sem_bordas
FROM default.gold_base_analitica
WHERE classe_codigo = 129 AND escopo_lei_atual
  AND dias_distribuicao_processamento IS NOT NULL;

**Leitura da sensibilidade:** 2,1% dos casos tocam o piso (15 dias) e 0,3%
o teto (180) — 2,4% somados. Excluindo as bordas, a mediana move de 48 →
49 e a média de 127,3 → 125,6: alteração marginal. A janela captura
comportamento natural, não trunca a distribuição — a métrica não depende
dos limites arbitrados.

**Resposta:** o tempo **mediano** entre distribuição e concessão do
processamento é de **48 dias** (~7 semanas); o P90 é de **149 dias**. A
média de **127 dias** é quase o triplo da mediana — cauda longa de demoras
que não desloca o tempo típico do sistema, mas revela alta variabilidade
entre juízos.

#### Resposta final (pergunta 1)

O tempo mediano entre a distribuição do pedido e a concessão do
processamento da RJ é de **48 dias** — cerca de 7 semanas. O percentil
90 é de **149 dias** (~5 meses): 1 em cada 10 recuperandas espera
mais do que isso. A média de **127 dias** é quase o triplo da mediana,
refletindo uma cauda longa de casos com demora muito acima do típico —
a característica de interesse para a jurimetria: o *tempo típico do
sistema* é razoável, mas a *incerteza* entre os processos varia
amplamente.

Metodologia: como a TPU do CNJ não reserva movimento próprio para o
deferimento do processamento (art. 52 da LRE), o marco é caracterizado
pela cascata R9 (despacho textual) → TPU 12444 (decisão de deferimento)
→ proxy "Mero expediente" (TPU 11010, janela 15–180 dias), conforme
detalhado no Bloco 4 do notebook 03. A janela do proxy foi validada
por sensibilidade: 2,4% de efeito de borda, sem impacto na mediana.

**Coerência com as perguntas seguintes:** o marco de concessão validado aqui é a fundação do conjunto — a pergunta 2 o usa como início do stay period, a pergunta 4 compara a fase pré-meritória entre varas com esta mesma métrica (com a mesma higiene de duração ≥ 0, confirmada pela soma de −8.359 dias dos 7 negativos), e a pergunta 5 o usa como denominador da taxa condicionada (2.422). A cobertura de 42,4% declarada nesta pergunta é, portanto, a cobertura herdada por todas as demais.